<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# Sedona quickstart

Spatial analytics in ten cells. We answer:

> **Which countries have the most airports per million people?**

Along the way we read two shapefiles, run a distributed spatial join, aggregate, write **GeoParquet 1.1** with auto covering-bbox metadata, read it back, and render an interactive map with **SedonaKepler**.

Data ships with the docker image (Natural Earth 50m admin and airports). No network required.

Once you've finished here, head to:

- `01-mobility-pulse.ipynb` — vector at scale: NYC taxi pickups by hour and neighborhood.
- `02-vegetation-change.ipynb` — full raster pipeline: NDVI change detection between two Sentinel-2 dates.
- `03-fire-risk-fusion.ipynb` — raster + vector fusion: wildfire risk score per building.
- `04-flood-snapshot.ipynb` — STAC + Sentinel-1 to flag buildings inside a recent flood extent.
- `05-geopandas-on-spark.ipynb` — scale your GeoPandas notebook to billions of rows.

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

In [ ]:
# Read country polygons. The shapefile reader is a first-class Spark V2 datasource
# in Sedona; no manual binary parsing.
countries = (
    sedona.read.format("shapefile")
    .load("data/ne_50m_admin_0_countries_lakes")
    .selectExpr(
        "geometry",
        "NAME as country",
        "CONTINENT as continent",
        "POP_EST as population",
    )
)
countries.show(5, truncate=40)

In [ ]:
airports = (
    sedona.read.format("shapefile")
    .load("data/ne_50m_airports")
    .selectExpr(
        "geometry",
        "name as airport",
        "type as airport_type",
    )
)
airports.show(5, truncate=40)

In [ ]:
# Spatial join: which country contains each airport?
# Sedona inserts a broadcast spatial index when one side is small.
airports.createOrReplaceTempView("airports")
countries.createOrReplaceTempView("countries")

joined = sedona.sql("""
    SELECT c.country, c.continent, c.population, a.airport
    FROM countries c
    JOIN airports a ON ST_Contains(c.geometry, a.geometry)
""")
joined.show(5, truncate=40)

In [ ]:
# Rank countries by airports per million people.
joined.createOrReplaceTempView("joined")

ranked = sedona.sql("""
    SELECT
        country,
        continent,
        population,
        COUNT(*)                                          AS airport_count,
        ROUND(COUNT(*) * 1e6 / NULLIF(population, 0), 2)  AS airports_per_million
    FROM joined
    WHERE population > 1000000
    GROUP BY country, continent, population
    ORDER BY airports_per_million DESC
""")
ranked.show(10)

In [ ]:
# Attach the rank back onto country geometries and write GeoParquet 1.1.
# Sedona auto-populates the covering-bbox metadata and projjson CRS from the
# geometry SRID, so downstream readers can prune partitions efficiently.
import os
import shutil

ranked.createOrReplaceTempView("ranked")
ranked_geo = sedona.sql("""
    SELECT c.country, c.continent,
           r.airport_count, r.airports_per_million,
           c.geometry
    FROM ranked r
    JOIN countries c USING (country)
""")

out_path = "/tmp/airports_per_country.parquet"
if os.path.exists(out_path):
    shutil.rmtree(out_path)
ranked_geo.coalesce(1).write.format("geoparquet").save(out_path)

sedona.read.format("geoparquet").load(out_path).show(5, truncate=40)

In [ ]:
from sedona.spark import SedonaKepler

ranked_geo_df = sedona.read.format("geoparquet").load(out_path)
SedonaKepler.create_map(df=ranked_geo_df, name="airports_per_million")

## What just happened?

In a handful of cells we ran an end-to-end spatial workflow:

1. **Read** two shapefiles directly into Spark DataFrames via Sedona's V2 shapefile datasource.
2. **Joined** them spatially with `ST_Contains` — Sedona inserts a broadcast spatial index automatically when one side is small.
3. **Aggregated and ranked** countries by airports per capita.
4. **Wrote** the result as **GeoParquet 1.1**; covering-bbox metadata and projjson CRS were populated automatically from the geometry SRID. Downstream readers can prune to a bounding box without scanning every row.
5. **Rendered** an interactive choropleth with **SedonaKepler**.

The exact same code path scales to billions of rows on a real Spark cluster — no API change, just more workers. Open `01-mobility-pulse.ipynb` next to see Sedona handle volume.